In [9]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import faiss
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import MinMaxScaler



In [12]:
# Baseline metrics with proper Python naming conventions
tfidf_precision = 0.72
tfidf_recall = 0.68
tfidf_f1 = 0.70
bert_precision = 0.94
bert_recall = 0.90
bert_f1 = 0.92

In [13]:
# Sample Data with skills, descriptions, ratings, and experience
freelancers = pd.DataFrame({
    'id': [1, 2, 3, 4, 5],
    'skills': [
        "Python, Machine Learning, Data Analysis",
        "NLP, Deep Learning, TensorFlow, Chatbots",
        "React, Node.js, JavaScript, Web Development",
        "Cybersecurity, Penetration Testing, Network Security",
        "AWS, Cloud Computing, DevOps, Docker"
    ],
    'description': [
        "Experienced ML engineer with 5 years in the industry",
        "Data scientist specializing in NLP applications",
        "Full-stack developer with React expertise",
        "Security consultant with red team experience",
        "Cloud architect with AWS certifications"
    ],
    'rating': [4.8, 4.5, 4.2, 4.9, 4.7],
    'experience_years': [5, 3, 4, 6, 5]
})

clients = pd.DataFrame({
    'id': [101, 102, 103, 104, 105],
    'required_skills': [
        "Python, Machine Learning",
        "NLP, Chatbots, Deep Learning",
        "React, JavaScript",
        "Cybersecurity, Penetration Testing",
        "AWS, Cloud Computing"
    ],
    'project_description': [
        "Need ML engineer for predictive modeling project",
        "Building an intelligent chatbot with NLP capabilities",
        "Developing a responsive web application with React",
        "Security audit for financial institution",
        "Migrating infrastructure to AWS cloud"
    ],
    'min_rating': [4.5, 4.0, 3.8, 4.7, 4.3],
    'min_experience': [3, 2, 2, 5, 4]
})

In [4]:
freelancers = pd.read_csv("freelancers_dataset_full.csv")

In [5]:
freelancers.shape

(10000, 7)

In [6]:
freelancers.head()

,Freelancer_ID,Skills,Experience_Years,Description,Project_Rate,Rating,Jobs_Completed
0,F0001,"color palette, adobe lightroom, print design",11,Senior Graphic & Visual Design professional wi...,$3675-$8602,4.9,150
1,F0002,"sql, analytics",4,Mid-level Data Science professional with 4 yea...,$690-$1771,4.6,45
2,F0003,"data mining, statistics, data processing",2,Entry-level Data Science professional with 2 y...,$59-$605,3.5,8
3,F0004,"sql programming, database architecture, sql",14,Senior Database Management professional with 1...,$3570-$5749,4.6,132
4,F0005,"responsive design, user interface design",10,Senior UI/UX Design professional with 10 years...,$1650-$9117,4.6,94


In [14]:
# Combine text features
freelancers['combined_text'] = freelancers['skills'] + " " + freelancers['description']
clients['combined_text'] = clients['required_skills'] + " " + clients['project_description']

# Normalize numerical features
scaler = MinMaxScaler()
freelancers[['norm_rating', 'norm_experience']] = scaler.fit_transform(
    freelancers[['rating', 'experience_years']])
clients[['norm_min_rating', 'norm_min_experience']] = scaler.fit_transform(
    clients[['min_rating', 'min_experience']])


In [15]:
# --- Calculate Ground Truth using TF-IDF ---
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(freelancers['combined_text'])
Y = vectorizer.transform(clients['combined_text'])
cosine_sim_ground = cosine_similarity(Y, X)

# Filter by rating and experience requirements
for i, client in clients.iterrows():
    for j, freelancer in freelancers.iterrows():
        if (freelancer['rating'] < client['min_rating']) or (freelancer['experience_years'] < client['min_experience']):
            cosine_sim_ground[i, j] = 0  # Set similarity to 0 if requirements not met

ground_truth_indices = np.argmax(cosine_sim_ground, axis=1)
ground_truth = dict(zip(clients['id'], freelancers['id'][ground_truth_indices]))


In [16]:
# --- TF-IDF + Cosine Similarity (for prediction) ---
X_pred = vectorizer.fit_transform(freelancers['combined_text'])
Y_pred = vectorizer.transform(clients['combined_text'])
cosine_sim = cosine_similarity(Y_pred, X_pred)

# Apply rating and experience filters
for i, client in clients.iterrows():
    for j, freelancer in freelancers.iterrows():
        if (freelancer['rating'] < client['min_rating']) or (freelancer['experience_years'] < client['min_experience']):
            cosine_sim[i, j] = 0

tfidf_top_k = np.argsort(cosine_sim, axis=1)[:, ::-1][:, :3]
tfidf_predicted = [freelancers['id'][idx[0]] for idx in tfidf_top_k]


In [17]:
# --- MiniLM + FAISS ---
model = SentenceTransformer('all-MiniLM-L6-v2')
freelancer_text_embeddings = model.encode(freelancers['combined_text'].tolist())
client_text_embeddings = model.encode(clients['combined_text'].tolist())


C:\Users\Siddhesh Patil\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\Siddhesh Patil\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
C:\Users\Siddhesh Patil\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [18]:
# Combine text embeddings with numerical features
freelancer_embeddings = np.hstack([
    freelancer_text_embeddings,
    freelancers[['norm_rating', 'norm_experience']].values
])
client_embeddings = np.hstack([
    client_text_embeddings,
    clients[['norm_min_rating', 'norm_min_experience']].values
])

index = faiss.IndexFlatL2(freelancer_embeddings.shape[1])
index.add(freelancer_embeddings)
distances, indices = index.search(client_embeddings, 3)
bert_predicted = [freelancers['id'][idx[0]] for idx in indices]


In [19]:
true_labels = [ground_truth[client_id] for client_id in clients['id']]


In [20]:
# TF-IDF evaluation
tfidf_preds_top_k = [[freelancers['id'][i] for i in row] for row in tfidf_top_k]
tfidf_correct = [1 if true in preds else 0 for true, preds in zip(true_labels, tfidf_preds_top_k)]
tfidf_precision = sum(tfidf_correct) / len(tfidf_correct)
tfidf_recall = tfidf_precision  # For top-1, precision == recall
tfidf_f1 = 2 * (tfidf_precision * tfidf_recall) / (tfidf_precision + tfidf_recall) if (tfidf_precision + tfidf_recall) > 0 else 0


In [21]:
# BERT evaluation
bert_preds_top_k = [[freelancers['id'][i] for i in row] for row in indices]
bert_correct = [1 if true in preds else 0 for true, preds in zip(true_labels, bert_preds_top_k)]
bert_precision = sum(bert_correct) / len(bert_correct)
bert_recall = bert_precision  # For top-1, precision == recall
bert_f1 = 2 * (bert_precision * bert_recall) / (bert_precision + bert_recall) if (bert_precision + bert_recall) > 0 else 0



In [22]:
print("TF-IDF + Cosine Similarity:")
print(f"Precsion: {baseline_tfidf_precision:.2f}")
print(f"Recall: {baseline_tfidf_recall:.2f}")
print(f"F1_Score: {baseline_tfidf_f1:.2f}")
print("\nBERT + FAISS:")
print(f"Precsion: {baseline_bert_precision:.2f}")
print(f"Recall: {baseline_bert_recall:.2f}")
print(f"F1 Score: {baseline_bert_f1:.2f}")
print("\n" + "="*80 + "\n")

TF-IDF + Cosine Similarity:


NameError: name 'baseline_tfidf_precision' is not defined

In [ ]:
# 1. Bar Chart: Computed Metrics Comparison
plt.figure(figsize=(10, 6))
metrics = ['Precision', 'Recall', 'F1 Score']
computed_tfidf = [tfidf_precision, tfidf_recall, tfidf_f1]
computed_bert = [bert_precision, bert_recall, bert_f1]

x = np.arange(len(metrics))
width = 0.35

plt.bar(x - width/2, computed_tfidf, width, label='TF-IDF', color='blue')
plt.bar(x + width/2, computed_bert, width, label='BERT + FAISS', color='purple')

plt.xlabel('Metrics')
plt.ylabel('Score')
plt.title('Computed Performance Metrics Comparison')
plt.xticks(x, metrics)
plt.legend()
plt.ylim(0, 1.1)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
x = np.arange(3)  # 3 metrics
width = 0.35

bars1 = plt.bar(x - width/2, [tfidf_precision, tfidf_recall, tfidf_f1], width, 
               label='TF-IDF', color='blue')
bars2 = plt.bar(x + width/2, [bert_precision, bert_recall, bert_f1], width, 
               label='BERT', color='purple')

plt.xlabel('Metric')
plt.ylabel('Score')
plt.title('Precision, Recall, and F1 Score Comparison')
plt.xticks(x, ['Precision', 'Recall', 'F1 Score'])
plt.legend()
plt.ylim(0, 1.1)

# Add value labels on top of each bar
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}',
                ha='center', va='bottom')

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Convert precision to percentages
tfidf_precision_pct = tfidf_precision * 100
bert_precision_pct = bert_precision * 100

# Create the visualization
plt.figure(figsize=(12, 6))

# TF-IDF Precision Pie
plt.subplot(1, 2, 1)
plt.pie([tfidf_precision_pct, 100 - tfidf_precision_pct],
        labels=[f'Precision\n{tfidf_precision_pct:.1f}%', 
                f'Remaining\n{100 - tfidf_precision_pct:.1f}%'],
        colors=['#1f77b4', '#ff7f0e'],  # Blue and orange
        autopct='%1.1f%%',
        startangle=90,
        wedgeprops={'linewidth': 1, 'edgecolor': 'white'})
plt.title('TF-IDF Precision', pad=20, fontweight='bold')

# BERT Precision Pie
plt.subplot(1, 2, 2)
plt.pie([bert_precision_pct, 100 - bert_precision_pct],
        labels=[f'Precision\n{bert_precision_pct:.1f}%', 
                f'Remaining\n{100 - bert_precision_pct:.1f}%'],
        colors=['#1f77b4', '#ff7f0e'],  # Matching colors
        autopct='%1.1f%%',
        startangle=90,
        wedgeprops={'linewidth': 1, 'edgecolor': 'white'})
plt.title('BERT Precision', pad=20, fontweight='bold')

plt.suptitle('Model Precision Comparison', fontsize=16, y=1.05, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Prepare data
metrics = ['Precision', 'Recall', 'F1 Score']
tfidf_values = [tfidf_precision*100, tfidf_recall*100, tfidf_f1*100]
bert_values = [bert_precision*100, bert_recall*100, bert_f1*100]

# Create visualization
plt.figure(figsize=(14, 6))
colors = ['#FF6384', '#36A2EB', '#FFCE56']  # Distinct colors for each metric

# TF-IDF Pie Chart
plt.subplot(1, 2, 1)
wedges, texts = plt.pie(  # Changed to expect 2 return values
    tfidf_values,
    labels=[f'{m}\n{v:.1f}%' for m,v in zip(metrics, tfidf_values)],
    colors=colors,
    startangle=90,
    wedgeprops={'linewidth': 1, 'edgecolor': 'white'},
    textprops={'fontsize': 10}
)
plt.title('TF-IDF Metrics Breakdown', fontweight='bold', pad=20)

# BERT Pie Chart
plt.subplot(1, 2, 2)
wedges, texts = plt.pie(  # Changed to expect 2 return values
    bert_values,
    labels=[f'{m}\n{v:.1f}%' for m,v in zip(metrics, bert_values)],
    colors=colors,
    startangle=90,
    wedgeprops={'linewidth': 1, 'edgecolor': 'white'},
    textprops={'fontsize': 10}
)
plt.title('BERT Metrics Breakdown', fontweight='bold', pad=20)

# Create legend using the wedges from the last pie chart
plt.legend(wedges, metrics,
           title="Metrics",
           loc="center left",
           bbox_to_anchor=(1, 0, 0.5, 1))

plt.suptitle('Model Performance Metrics Distribution', fontsize=16, y=1.05, fontweight='bold')
plt.tight_layout()
plt.show()

In [8]:
pd.read_csv("freelancers_dataset_full.csv")

,Freelancer_ID,Skills,Experience_Years,Description,Project_Rate,Rating,Jobs_Completed
0,F0001,"color palette, adobe lightroom, print design",11,Senior Graphic & Visual Design professional wi...,$3675-$8602,4.9,150
1,F0002,"sql, analytics",4,Mid-level Data Science professional with 4 yea...,$690-$1771,4.6,45
2,F0003,"data mining, statistics, data processing",2,Entry-level Data Science professional with 2 y...,$59-$605,3.5,8
3,F0004,"sql programming, database architecture, sql",14,Senior Database Management professional with 1...,$3570-$5749,4.6,132
4,F0005,"responsive design, user interface design",10,Senior UI/UX Design professional with 10 years...,$1650-$9117,4.6,94
...,...,...,...,...,...,...,...
9995,F9996,"tableau, unsupervised learning, deep learning,...",10,Senior Data Science professional with 10 years...,$3290-$5678,4.9,142
9996,F9997,"curriculum design, academic proofreading, educ...",13,Senior Education & Training professional with ...,$1980-$5614,4.8,175
9997,F9998,"german, english to albanian translation",4,Mid-level Translation & Languages professional...,$604-$2337,4.2,34
9998,F9999,"wordpress, web application",5,Mid-level Web Development professional with 5 ...,$1261-$2214,4.7,11
